In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

# 1. 设置路径与全局参数
sc.set_figure_params(figsize=(5, 5), dpi=100)
data_dir = r"D:\DATA\cache1_h5ad" 
out_dir = r"D:\DATA\results"      
os.makedirs(out_dir, exist_ok=True)

# 2. 读取并过滤样本
h5ad_files = glob.glob(os.path.join(data_dir, "*.h5ad"))
print(f"找到 {len(h5ad_files)} 个 h5ad 文件")

adata_list = []
batch_keys = [] 

for file in h5ad_files:
    sample_name = os.path.basename(file).replace(".h5ad", "")
    if "PRIMARY_TUMOUR" in sample_name.upper() or "NORMAL" in sample_name.upper():
        temp_adata = sc.read_h5ad(file)
        
        # 标注分组
        if "NORMAL" in sample_name.upper():
            temp_adata.obs['Group'] = 'Non-tumour'
        else:
            temp_adata.obs['Group'] = 'Primary Tumor'
            
        temp_adata.obs['Sample'] = sample_name
        adata_list.append(temp_adata)
        batch_keys.append(sample_name)

# 3. 合并数据
adata = ad.concat(adata_list, label="batch", keys=batch_keys, index_unique="-")
print(adata)
print(adata.obs['Group'].value_counts())

In [ ]:
# 1. 基础过滤
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# 2. 计算线粒体比例
adata.var['mt'] = adata.var_names.str.startswith('MT-')
adata.var['mt'] = adata.var['mt'].to_numpy() 
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# 3. 过滤高线粒体细胞并固定矩阵
adata = adata[adata.obs.pct_counts_mt < 15, :].copy()

# 4. 标准化与保存原始数据
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata 

# 5. 降维与特征选择
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, batch_key='batch')
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
import harmonypy

# 1. 直接调用 harmonypy，绕开 scanpy 旧版封装
print("开始运行 Harmony 去批次...")
ho = harmonypy.run_harmony(adata.obsm['X_pca'], adata.obs, ['batch'])

# 2. 获取结果并强制转为 numpy 数组（避免 getnnz 底层报错）
z_corr = ho.Z_corr
if z_corr.shape[0] != adata.n_obs:
    z_corr = z_corr.T
adata.obsm['X_pca_harmony'] = np.asarray(z_corr, dtype=np.float32)
print(f"Harmony 完成，矩阵形状: {adata.obsm['X_pca_harmony'].shape}")

# 3. 构建邻接图与 UMAP
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['Group', 'batch'], wspace=0.4)

In [ ]:
# 1. Leiden 聚类 (加入 flavor 参数消除警告)
sc.tl.leiden(adata, resolution=0.6, key_added='leiden', flavor="igraph", n_iterations=2, directed=False)
sc.pl.umap(adata, color=['leiden'], legend_loc='on data')

# 2. CellTypist 自动注释
import celltypist
from celltypist import models

print("正在加载模型进行注释...")
model = models.Model.load(model='Immune_All_Low.pkl') 
predictions = celltypist.annotate(adata, model=model, majority_voting=True)
adata.obs['cell_type'] = predictions.predicted_labels.majority_voting.values
# 3.
cell_type_mapping = {
    'Tem/Effector helper T cells': 'T cells', 'Regulatory T cells': 'T cells',
    'Tem/Temra cytotoxic T cells': 'T cells', 'Tem/Trm cytotoxic T cells': 'T cells', 'Trm cytotoxic T cells': 'T cells',
    'CD16+ NK cells': 'NK cells', 'CD16- NK cells': 'NK cells', 'NK cells': 'NK cells',
    'Classical monocytes': 'Macrophages/Monocytes', 'Non-classical monocytes': 'Macrophages/Monocytes',
    'Alveolar macrophages': 'Macrophages/Monocytes', 'Macrophages': 'Macrophages/Monocytes',
    'DC': 'Dendritic cells', 'DC1': 'Dendritic cells', 'pDC': 'Dendritic cells',
    'Memory B cells': 'B cells'
}
adata.obs['cell_type_major'] = adata.obs['cell_type'].replace(cell_type_mapping)

# 4. 绘制美化后的 UMAP
sc.pl.umap(
    adata, color=['cell_type_major'], legend_loc='right margin', frameon=False,
    legend_fontsize=12, title='Major Cell Types', wspace=0.4, palette='tab20',
    save='_celltype_annotation.pdf'
)

In [ ]:
# 1. 提取全基因矩阵，附加最新注释
adata_full = adata.raw.to_adata()
adata_full.obs = adata.obs 

# 2. 绘制细胞比例堆叠柱状图
cell_type_counts = adata_full.obs.groupby(['Sample', 'Group', 'cell_type_major'], observed=True).size().unstack(fill_value=0)
cell_type_fracs = cell_type_counts.div(cell_type_counts.sum(axis=1), axis=0)
plot_df = cell_type_fracs.reset_index().melt(id_vars=['Sample', 'Group'], var_name='CellType', value_name='Fraction')

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x='Sample', y='Fraction', hue='CellType', dodge=False)
plt.xticks(rotation=90)
plt.title('Cell Type Fractions (Primary Tumor vs Non-tumour)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'cell_type_fractions.pdf'))

# 3. 差异基因分析
sc.tl.rank_genes_groups(adata_full, 'Group', method='wilcoxon', reference='Non-tumour')
degs_df = sc.get.rank_genes_groups_df(adata_full, group='Primary Tumor')
degs_df.to_csv(os.path.join(out_dir, 'DEGs_Primary_vs_NonTumour.csv'), index=False)

# 4. 绘制 NC 风格的火山图
df = degs_df.copy()
df['log2FC'] = df['logfoldchanges']
df['-log10(adj.P-val)'] = -np.log10(df['pvals_adj'])
df['significance'] = 'Not Significant'
df.loc[(df['log2FC'] > 1.0) & (df['pvals_adj'] < 0.05), 'significance'] = 'Up'
df.loc[(df['log2FC'] < -1.0) & (df['pvals_adj'] < 0.05), 'significance'] = 'Down'

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='log2FC', y='-log10(adj.P-val)', hue='significance',
                palette={'Up': 'red', 'Down': 'blue', 'Not Significant': 'grey'}, alpha=0.7)

# 标注 Top 基因
top_genes = pd.concat([df[df['significance']=='Up'].nsmallest(5, 'pvals_adj')['names'],
                       df[df['significance']=='Down'].nsmallest(5, 'pvals_adj')['names']])
for _, row in df[df['names'].isin(top_genes)].iterrows():
    plt.annotate(row['names'], (row['log2FC'], row['-log10(adj.P-val)']), fontsize=8, ha='center')

plt.axhline(-np.log10(0.05), color='grey', linestyle='--', linewidth=0.8)
plt.axvline(1.0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(-1.0, color='grey', linestyle='--', linewidth=0.8)
plt.xlabel('log2 Fold Change')
plt.ylabel('-log10(adjusted P-value)')
plt.title('Volcano Plot: Primary Tumor vs Non-tumour')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'volcano_plot.pdf'), dpi=300)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. 干净地重新计算数据
df_plot = cell_type_fracs.reset_index()
group_order = {'Non-tumour': 0, 'Primary Tumor': 1}
df_plot['GroupOrder'] = df_plot['Group'].map(group_order)
df_plot = df_plot.sort_values(['GroupOrder', 'Sample'], ascending=[True, True])

# 2. 核心优化：提取并缩短样本名字
# 把 GSM3516674_MSK_LX684_PRIMARY_TUMOUR 变成 LX684_T
def shorten_name(name):
    if 'PRIMARY_TUMOUR' in name:
        return name.split('_')[2] + '_T'
    elif 'NORMAL' in name:
        return name.split('_')[2] + '_N'
    return name

df_plot['ShortName'] = df_plot['Sample'].apply(shorten_name)
df_plot = df_plot.set_index('ShortName')

cell_types = [c for c in df_plot.columns if c not in ['Sample', 'Group', 'GroupOrder']]
df_bar = df_plot[cell_types]

# 3. 画图
fig, ax = plt.subplots(figsize=(10, 8))
df_bar.plot(kind='barh', stacked=True, ax=ax, width=0.8, 
            edgecolor='white', linewidth=0.5, colormap='tab20')

# 4. 美化坐标轴
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.set_xlabel('Fraction', fontsize=12)
ax.set_ylabel('')
ax.set_title('Cell Type Fractions', loc='left', fontsize=14, fontweight='bold')

ax.yaxis.tick_right()
ax.tick_params(axis='y', length=0, labelsize=11)
ax.set_xlim(0, 1.0)

# 5. 在左侧画分组黑线和组名
samples = df_bar.index.tolist()
for group in ['Non-tumour', 'Primary Tumor']:
    group_samples = df_plot[df_plot['Group'] == group].index.tolist()
    if not group_samples: continue
    
    start_idx = samples.index(group_samples[0])
    end_idx = samples.index(group_samples[-1])
    mid_idx = (start_idx + end_idx) / 2.0
    
    ax.plot([-0.02, -0.02], [start_idx - 0.4, end_idx + 0.4], 
            color='black', clip_on=False, lw=2, transform=ax.get_yaxis_transform())
    
    ax.text(-0.06, mid_idx, group, rotation=90, va='center', ha='right', 
            fontsize=13, fontweight='bold', transform=ax.get_yaxis_transform())

# 6. 图例位置微调
ax.legend(bbox_to_anchor=(1.25, 1), loc='upper left', title='Cell Type', 
          frameon=False, fontsize=10, title_fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'vertical_barplot_fixed.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import zscore
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 提取 Top 差异基因
top_up = degs_df[degs_df['logfoldchanges'] > 0].nsmallest(25, 'pvals_adj')['names'].tolist()
top_down = degs_df[degs_df['logfoldchanges'] < 0].nsmallest(25, 'pvals_adj')['names'].tolist()
top_genes = top_up + top_down

# 2. 提取表达矩阵
adata_heat = adata_full[:, top_genes].copy()

# 3. 按组随机抽样 150 个细胞
np.random.seed(42)
cells_norm = adata_heat.obs[adata_heat.obs['Group'] == 'Non-tumour'].sample(n=150, random_state=42).index
cells_tumor = adata_heat.obs[adata_heat.obs['Group'] == 'Primary Tumor'].sample(n=150, random_state=42).index
sampled_cells = list(cells_norm) + list(cells_tumor)

adata_sub = adata_heat[sampled_cells, :].copy()
expr_matrix = adata_sub.X.toarray() if sp.issparse(adata_sub.X) else adata_sub.X
df_heat = pd.DataFrame(expr_matrix, columns=top_genes, index=sampled_cells).T

# 4. Z-score 标准化并截断
scaled_expr = df_heat.apply(zscore, axis=1)
scaled_expr = np.clip(scaled_expr, -2, 2)

# 5. 准备顶部注释条
group_labels = ['Non-tumour'] * 150 + ['Primary Tumor'] * 150
col_colors = pd.Series(group_labels, index=sampled_cells).map({
    'Non-tumour': '#4DBBD5',  # 经典蓝色
    'Primary Tumor': '#E64B35' # 经典红色
})

n_norm = adata_full.obs[adata_full.obs['Group'] == 'Non-tumour']['Sample'].nunique()
n_tumor = adata_full.obs[adata_full.obs['Group'] == 'Primary Tumor']['Sample'].nunique()

# 6. 绘制热图
fig = plt.figure(figsize=(8, 9)) # 设置整个画布大小
# 指定网格布局，给底部的 colorbar 留出空间
gs = fig.add_gridspec(2, 2, height_ratios=[0.05, 0.95], hspace=0.05, wspace=0.05)

g = sns.clustermap(
    scaled_expr,
    cmap='RdBu_r',          
    col_colors=col_colors,      
    row_cluster=False,          
    col_cluster=False,          
    xticklabels=False,          
    yticklabels=scaled_expr.index,
    figsize=(8, 9),
    linewidths=0,               
    cbar_pos=None               
)

# 7. 调整坐标轴位置
# 隐藏左侧和顶部的多余树状图（我们没聚类，但这些位置会占空间）
g.ax_row_dendrogram.set_visible(False)
g.ax_col_dendrogram.set_visible(False)

# 强制把基因名字放在左侧，并靠紧热图
g.ax_heatmap.yaxis.set_ticks_position('left')
g.ax_heatmap.yaxis.set_label_position('left')
# 调整热图本身的位置（左、底、宽、高）
g.ax_heatmap.set_position([0.2, 0.1, 0.7, 0.75]) 

# 8. 在底部手动添加 Colorbar
# 创建一个新的坐标轴 [左侧起始位置, 底部起始位置, 宽度, 高度]
cbar_ax = g.fig.add_axes([0.3, 0.05, 0.3, 0.015]) 
cb = plt.colorbar(g.ax_heatmap.collections[0], cax=cbar_ax, orientation='horizontal')
cb.set_label('Expression Z-score', fontsize=12, fontweight='bold', labelpad=5)
cb.ax.tick_params(labelsize=10)

# 9. 添加顶部标签
g.ax_heatmap.text(0.25, 1.02, f'Non-tumour\n(n={n_norm})', 
                  ha='center', va='bottom', transform=g.ax_heatmap.transAxes, 
                  fontsize=12, fontweight='bold')
g.ax_heatmap.text(0.75, 1.02, f'Primary Tumor\n(n={n_tumor})', 
                  ha='center', va='bottom', transform=g.ax_heatmap.transAxes, 
                  fontsize=12, fontweight='bold')

# 10. 美化细节
g.ax_heatmap.set_xlabel('')
g.ax_heatmap.set_ylabel('')
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=11, fontstyle='italic')

# 添加大标题
g.fig.suptitle('Top Differentially Expressed Genes', fontsize=16, fontweight='bold', y=0.95)

# 保存并显示
g.savefig(os.path.join(out_dir, 'DEG_heatmap_NCS_final.pdf'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import liana as li
import matplotlib.pyplot as plt

# 1. 筛选出高置信度的通讯结果 (magnitude_rank <= 0.01)
filtered_liana = adata_comm.uns['liana_res'].copy()
filtered_liana = filtered_liana[filtered_liana['magnitude_rank'] <= 0.01] 

# 2. 绘制圈图（直接出图，不接收返回值）
li.pl.circle_plot(
    adata=adata_comm,
    groupby='cell_type_major',
    liana_res=filtered_liana,
    score_key='magnitude_rank', 
    inverse_score=True,
    figure_size=(7, 7)
)

# 3. 用 matplotlib 的方式强行保存当前画布（以防万一）
plt.savefig(os.path.join(out_dir, 'cell_communication_circle.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 计算拟时序 (DPT)
# ==========================================
# 设定起点：这里默认用 'leiden' 聚类结果中的第 0 群作为起点
# (注意：在真实分析中，你应该根据 marker 基因选择一个代表生物学起点的 cluster)
if 'leiden' not in adata.obs.columns:
    print("未找到 leiden 聚类，重新运行聚类...")
    sc.tl.leiden(adata, resolution=0.6, key_added='leiden', flavor="igraph", n_iterations=2, directed=False)

# 确保使用去批次后的 PCA 空间
if 'X_pca_harmony' in adata.obsm:
    use_rep = 'X_pca_harmony'
else:
    use_rep = 'X_pca'

# 重新构建邻居图（如果前面的图被覆盖了，这里保证图是存在的）
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep=use_rep)

# 计算扩散图和拟时序
print("正在计算拟时序 (DPT)...")
sc.tl.diffmap(adata)
adata.uns['iroot'] = np.flatnonzero(adata.obs['leiden'] == '0')[0] # 设置起点细胞
sc.tl.dpt(adata)
print("拟时序计算完成！")

# ==========================================
# 2. 可视化对比 (小提琴图 / 密度图)
# ==========================================
# 确保颜色配置正确
my_palette = {'Non-tumour': '#4DBBD5', 'Primary Tumor': '#E64B35'}

# 绘制小提琴图
plt.figure(figsize=(8, 5))
sns.violinplot(data=adata.obs, x='Group', y='dpt_pseudotime', palette=my_palette, hue='Group', legend=False)
plt.title('Pseudotime Distribution by Group', fontsize=14, fontweight='bold')
plt.ylabel('DPT Pseudotime', fontsize=12)
plt.xlabel('')
plt.show()

# 绘制密度图 (KDE)
plt.figure(figsize=(8, 5))
sns.kdeplot(data=adata.obs[adata.obs['Group']=='Non-tumour'], x='dpt_pseudotime', label='Non-tumour', fill=True, color='#4DBBD5', alpha=0.5)
sns.kdeplot(data=adata.obs[adata.obs['Group']=='Primary Tumor'], x='dpt_pseudotime', label='Primary Tumor', fill=True, color='#E64B35', alpha=0.5)
plt.title('Pseudotime Density by Group', fontsize=14, fontweight='bold')
plt.xlabel('DPT Pseudotime', fontsize=12)
plt.ylabel('Density')
plt.legend()
plt.show()